In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, FFMpegWriter
from IPython.display import display, HTML
import time
import pickle
import os
import sys
import csv


In [6]:
## Boundary layer thickness
def calcular_boundary_layer_thickness(u, y, u0=1.0):
    perfil = u[:, u.shape[1] // 2]
    for i in range(len(perfil)):
        if perfil[i] >= 0.99 * u0:
            return y[i]
    return y[-1]  # En caso de que nunca alcance 99%

## Pressure coef
def calcular_coeficiente_presion_medio(p):
    return np.mean(p[:, 0])  # Promedio en la pared izquierda, por ejemplo

## Mean temperature
def calcular_temperatura_media(T):
    return np.mean(T)

# Orden de convergencia y GCI
def orden_convergencia(f1, f2, f3, r):
    return np.log((f3 - f2) / (f2 - f1)) / np.log(r)

def GCI(f1, f2, r, p, Fs=1.25):
    return Fs * abs(f1 - f2) / abs(f1) / (r**p - 1)

## Haller separation (pressure boundary layer)
### Introducir una "pressure-driven boundary layer" en la pared superior que provoque separación

In [3]:
def run_simulation_pressures_boundary_layer(nx, ny):
    # --- Parámetros adimensionales ---
    Re, Pr, Ec, Eu = 20.0, 10.0, 0.1, 1.0
    Lx_star, Ly_star = 1.0, 1.0
    dx_star = Lx_star / (nx - 1)
    dy_star = Ly_star / (ny - 1)
    up, u0 = -0.5, 1.0
    T0_star, T1_star = 0.0, 0.0

    # --- CFL-SAFE TIME STEP (STATIC) ---
    cfl_limit = 0.1
    u_max_expected = 2.0
    v_max_expected = 2.0
    dt_cfl = cfl_limit * min(dx_star / u_max_expected, dy_star / v_max_expected)
    dt_star = min(dt_cfl, 1e-3, 1.0 / 1100)  # garantiza que nt >= 1100 y t* <= 1.0
    nt = int(1.0 / dt_star)



    # --- Malla y campos ---
    x_star = np.linspace(0, Lx_star, nx)
    y_star = np.linspace(0, Ly_star, ny)
    X_star, Y_star = np.meshgrid(x_star, y_star)

    u_star = np.full((ny, nx), u0, dtype=np.float64)
    v_star = np.zeros((ny, nx), dtype=np.float64)
    p_star = np.zeros((ny, nx), dtype=np.float64)
    T_star = np.full((ny, nx), T1_star, dtype=np.float64)

    # --- Condiciones de frontera iniciales ---
    u_star[0, :] = up
    u_star[-1, :] = u0
    v_star[0, :] = v_star[-1, :] = 0
    T_star[0, :] = T0_star
    T_star[-1, :] = T1_star

    u_history, v_history, p_history, T_history, tau_history = [], [], [], [], []

    for n in range(nt):
        u_old, v_old, p_old, T_old = u_star.copy(), v_star.copy(), p_star.copy(), T_star.copy()

        # --- Pressure layer at y = 1 (top wall) ---
        A, B = 0.5, 20.0
        for j in range(1, nx - 1):
            x_loc = x_star[j]
            p_star[-1, j] = 1.0 - A * np.exp(-B * (x_loc - 0.5)**2)

        # --- Ecuaciones de momento ---
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_u_x = u_old[i, j] * (u_old[i, j + 1] - u_old[i, j - 1]) / (2 * dx_star)
                conv_u_y = v_old[i, j] * (u_old[i + 1, j] - u_old[i - 1, j]) / (2 * dy_star)
                diff_u = ((u_old[i, j + 1] - 2 * u_old[i, j] + u_old[i, j - 1]) / dx_star**2 +
                          (u_old[i + 1, j] - 2 * u_old[i, j] + u_old[i - 1, j]) / dy_star**2)
                u_star[i, j] = u_old[i, j] + dt_star * (-conv_u_x - conv_u_y + (1 / Re) * diff_u)

                conv_v_x = u_old[i, j] * (v_old[i, j + 1] - v_old[i, j - 1]) / (2 * dx_star)
                conv_v_y = v_old[i, j] * (v_old[i + 1, j] - v_old[i - 1, j]) / (2 * dy_star)
                diff_v = ((v_old[i, j + 1] - 2 * v_old[i, j] + v_old[i, j - 1]) / dx_star**2 +
                          (v_old[i + 1, j] - 2 * v_old[i, j] + v_old[i - 1, j]) / dy_star**2)
                grad_p_y = (p_old[i + 1, j] - p_old[i - 1, j]) / (2 * dy_star)
                v_star[i, j] = v_old[i, j] + dt_star * (-conv_v_x - conv_v_y - Eu * grad_p_y + (1 / Re) * diff_v)

        # --- Solver de presión ---
        for _ in range(200):
            for i in range(1, ny - 1):
                for j in range(1, nx - 1):
                    rhs = ((u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star) +
                           (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star))
                    p_star[i, j] = 0.25 * (p_old[i + 1, j] + p_old[i - 1, j] + p_old[i, j + 1] + p_old[i, j - 1] -
                                          (dx_star * dy_star) / (2 * (dx_star**2 + dy_star**2)) * rhs)

        # --- Corrección de velocidades ---
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                u_star[i, j] -= dt_star * Eu * (p_star[i, j + 1] - p_star[i, j - 1]) / (2 * dx_star)
                v_star[i, j] -= dt_star * Eu * (p_star[i + 1, j] - p_star[i - 1, j]) / (2 * dy_star)

        # --- Ecuación de energía ---
        for i in range(1, ny - 1):
            for j in range(1, nx - 1):
                conv_T_x = u_star[i, j] * (T_old[i, j + 1] - T_old[i, j - 1]) / (2 * dx_star)
                conv_T_y = v_star[i, j] * (T_old[i + 1, j] - T_old[i - 1, j]) / (2 * dy_star)
                diff_T = ((T_old[i, j + 1] - 2 * T_old[i, j] + T_old[i, j - 1]) / dx_star**2 +
                          (T_old[i + 1, j] - 2 * T_old[i, j] + T_old[i - 1, j]) / dy_star**2)
                Sxx = (u_star[i, j + 1] - u_star[i, j - 1]) / (2 * dx_star)
                Syy = (v_star[i + 1, j] - v_star[i - 1, j]) / (2 * dy_star)
                Sxy = 0.5 * ((u_star[i + 1, j] - u_star[i - 1, j]) / (2 * dy_star) +
                             (v_star[i, j + 1] - v_star[i, j - 1]) / (2 * dx_star))
                viscous_heating = (Ec / Re) * (2 * (Sxx**2 + Syy**2) + 4 * Sxy**2)
                T_star[i, j] = T_old[i, j] + dt_star * (-conv_T_x - conv_T_y + (1 / (Re * Pr)) * diff_T + viscous_heating)

        # --- Shear stress ---
        tau = np.zeros_like(u_star)
        tau[1:-1, :] = (u_star[2:, :] - u_star[:-2, :]) / (2 * dy_star)

        # --- Condiciones de frontera ---
        u_star[0, :] = up
        u_star[-1, :] = u0
        u_star[:, -1] = u_star[:, -2]
        v_star[0, :] = v_star[-1, :] = 0
        v_star[:, -1] = 0
        p_star[:, 0] = p_star[:, 1]
        p_star[:, -1] = p_star[:, -2]
        p_star[0, :] = p_star[1, :]
        p_star[-1, 0] = p_star[-1, 1]
        p_star[-1, -1] = p_star[-1, -2]

        if n % 10 == 0:
            print(f"\rPaso {n}/{nt} (dt*={dt_star:.5f})", end="")
            sys.stdout.flush()
            u_history.append(u_star.copy())
            v_history.append(v_star.copy())
            p_history.append(p_star.copy())
            T_history.append(T_star.copy())
            tau_history.append(tau.copy())

    print("\rSimulación finalizada.                      ")

    return {
        "u_history": u_history,
        "v_history": v_history,
        "p_history": p_history,
        "T_history": T_history,
        "tau_history": tau_history,
        "params": {"nx": nx, "ny": ny, "dt_star": dt_star, "nt": nt},
        "X_star": X_star,
        "Y_star": Y_star
    }


In [7]:
# --- Configuración de mallas y ejecución ---
resoluciones = [(25, 25), (50, 50)]
resultados = []

for nx, ny in resoluciones:
    sim = run_simulation_pressures_boundary_layer(nx, ny)
    u = sim["u_history"][-1]
    p = sim["p_history"][-1]
    T = sim["T_history"][-1]
    y = sim["Y_star"][:, 0]

    delta = calcular_boundary_layer_thickness(u, y)
    cp = calcular_coeficiente_presion_medio(p)
    Tavg = calcular_temperatura_media(T)

    resultados.append({
        "res": f"{nx}x{ny}",
        "delta": delta,
        "Cp": cp,
        "Tavg": Tavg
    })


Simulación finalizada.                      
Simulación finalizada.                      


In [ ]:
# --- Análisis de convergencia ---
f_delta = [r["delta"] for r in resultados]
f_cp = [r["Cp"] for r in resultados]
f_T = [r["Tavg"] for r in resultados]

r = 2  # relación entre mallas
p_delta = orden_convergencia(f_delta[2], f_delta[1], f_delta[0], r)
p_cp = orden_convergencia(f_cp[2], f_cp[1], f_cp[0], r)
p_T = orden_convergencia(f_T[2], f_T[1], f_T[0], r)

gci_delta = GCI(f_delta[2], f_delta[1], r, p_delta)
gci_cp = GCI(f_cp[2], f_cp[1], r, p_cp)
gci_T = GCI(f_T[2], f_T[1], r, p_T)

# --- Mostrar resultados ---
print("\nConvergencia de resolución:")
print("Resolución | Espesor δ | Cp medio | Temp media")
for r in resultados:
    print(f"{r['res']:>10} | {r['delta']:.5f} | {r['Cp']:.5f} | {r['Tavg']:.5f}")

print("\nOrdenes de convergencia:")
print(f"δ: {p_delta:.3f}, Cp: {p_cp:.3f}, Temp: {p_T:.3f}")

print("\nGCI (con Fs = 1.25):")
print(f"GCI_δ: {gci_delta:.3e}, GCI_Cp: {gci_cp:.3e}, GCI_T: {gci_T:.3e}")

# --- Gráficos (opcional) ---
res_labels = [r["res"] for r in resultados]

plt.figure()
plt.plot(res_labels, f_delta, marker='o')
plt.title("Espesor de capa límite vs resolución")
plt.ylabel("δ")
plt.xlabel("Resolución")
plt.grid()

plt.figure()
plt.plot(res_labels, f_cp, marker='s')
plt.title("Cp medio vs resolución")
plt.ylabel("Cp")
plt.xlabel("Resolución")
plt.grid()

plt.figure()
plt.plot(res_labels, f_T, marker='^')
plt.title("Temperatura media vs resolución")
plt.ylabel("T_media")
plt.xlabel("Resolución")
plt.grid()

plt.show()